In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

I0000 00:00:1780067118.340200  205236 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780067118.362127  205236 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# --- 1. Define the Molecule (The Graph) ---
print("Building the Molecular Graph for 1-Butanol (C-C-C-C-O)...")
num_nodes = 5

# Adjacency Matrix (A): Represents chemical bonds
A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
# The chain of bonds: C0-C1, C1-C2, C2-C3, C3-O4
atomslist = ["C0", "C1", "C2", "C3", "O4"]  # Atom labels for clarity
bonds = [(0, 1), (1, 2), (2, 3), (3, 4)] 

for i, j in bonds:
    A[i, j] = 1.0
    A[j, i] = 1.0 # Bonds go both ways

# Add "Self-Loops": Atoms retain their own features during message passing
np.fill_diagonal(A, 1.0)

# Normalize the Adjacency matrix
D = np.sum(A, axis=1)
A_norm = A / D[:, None]

print("Adjacency Matrix (A):")
print("    ", end="")
for i in range(num_nodes):
    print(f"{atomslist[i]:3s}", end=" ")
print()
for i in range(num_nodes):
    print(f"{atomslist[i]:3s}", end=" ")
    for j in range(num_nodes):
        print(f"{A[i, j]:3.1f}", end=" ")
    print()  

print("\nNormalized Adjacency Matrix (A_norm):")
print("      ", end="")
for i in range(num_nodes):
    print(f"{atomslist[i]:4s}", end=" ")
print()
for i in range(num_nodes):
    print(f"{atomslist[i]:4s}", end=" ")
    for j in range(num_nodes):
        print(f"{A_norm[i, j]:4.2f}", end=" ")
    print()           


Building the Molecular Graph for 1-Butanol (C-C-C-C-O)...
Adjacency Matrix (A):
    C0  C1  C2  C3  O4  
C0  1.0 1.0 0.0 0.0 0.0 
C1  1.0 1.0 1.0 0.0 0.0 
C2  0.0 1.0 1.0 1.0 0.0 
C3  0.0 0.0 1.0 1.0 1.0 
O4  0.0 0.0 0.0 1.0 1.0 

Normalized Adjacency Matrix (A_norm):
      C0   C1   C2   C3   O4   
C0   0.50 0.50 0.00 0.00 0.00 
C1   0.33 0.33 0.33 0.00 0.00 
C2   0.00 0.33 0.33 0.33 0.00 
C3   0.00 0.00 0.33 0.33 0.33 
O4   0.00 0.00 0.00 0.50 0.50 


In [3]:
# --- 2. Define Features and Labels ---
# Node Features (X): Random initial data. 
# We use random data to prove the GNN relies on the *Bonds* (Structure), not the features!
X = np.random.randn(num_nodes, 4).astype(np.float32)

# Labels: 0 = Hydrophobic (Carbon), 1 = Hydrophilic (Oxygen). 
# -1 means the GNN doesn't know what this atom is.
true_labels = np.array([0, -1, -1, -1, 1], dtype=np.int32)

# Mask: We only train the AI using Atom 0 and Atom 4
train_mask = np.array([True, False, False, False, True])

# Convert to TensorFlow tensors
A_tensor = tf.constant(A_norm)
X_tensor = tf.constant(X)
y_tensor = tf.constant(true_labels)

In [ ]:
# --- 3. Build the Graph Convolution Layer ---
class GraphConv(layers.Layer):
    # We add an activation parameter, defaulting to None
    def __init__(self, output_dim, activation=None):
        super().__init__()
        self.output_dim = output_dim
        self.activation = activation

    def build(self, input_shape):
        feature_dim = input_shape[0][-1]
        self.W = self.add_weight(shape=(feature_dim, self.output_dim),
                                 initializer="glorot_uniform", trainable=True)

    def call(self, inputs):
        X_in, A_in = inputs
        # 1. Transform atomic features
        transformed_features = tf.matmul(X_in, self.W)
        
        # 2. MESSAGE PASSING: Sum up features from bonded neighbor atoms
        message_passed = tf.matmul(A_in, transformed_features)
        
        # Apply activation ONLY if one was provided
        if self.activation is not None:
            return self.activation(message_passed)
        return message_passed

In [ ]:
# --- 4. Build the GNN Architecture  ---
print("Building the Molecular GNN...")
features_input = layers.Input(shape=(4,))
adj_input = layers.Input(shape=(num_nodes,))

# The hidden layer uses ReLU to learn complex patterns
h = GraphConv(8, activation=tf.nn.relu)([features_input, adj_input])

# THE FIX: The final output layer has NO activation (Linear)
output = GraphConv(2, activation=None)([h, adj_input]) 

gnn_model = models.Model(inputs=[features_input, adj_input], outputs=output)
# --- 5. Train the GNN ---
optimizer = optimizers.Adam(learning_rate=0.05)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

print("Training the GNN to learn molecular properties...")
for epoch in range(100):
    with tf.GradientTape() as tape:
        logits = gnn_model([X_tensor, A_tensor])
        
        # Mask out the unknown middle atoms during training
        masked_logits = tf.boolean_mask(logits, train_mask)
        masked_labels = tf.boolean_mask(y_tensor, train_mask)
        
        loss = loss_fn(masked_labels, masked_logits)
        
    grads = tape.gradient(loss, gnn_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, gnn_model.trainable_variables))

Building the Molecular GNN...
Training the GNN to learn molecular properties...


In [7]:
# --- 6. The Final Reveal ---
print("\n--- Final Atom Classifications ---")
final_logits = gnn_model([X_tensor, A_tensor])
predictions = tf.argmax(tf.nn.softmax(final_logits), axis=1).numpy()

atom_names = ["Carbon 0", "Carbon 1", "Carbon 2", "Carbon 3", "Oxygen 4"]

for i in range(num_nodes):
    status = "  (Known Label)" if train_mask[i] else " (AI Predicted)"
    property_type = "Hydrophobic (Non-Polar)" if predictions[i] == 0 else "Hydrophilic (Polar)"
    print(f"{atom_names[i]}{status}: {property_type}")


--- Final Atom Classifications ---
Carbon 0  (Known Label): Hydrophobic (Non-Polar)
Carbon 1 (AI Predicted): Hydrophobic (Non-Polar)
Carbon 2 (AI Predicted): Hydrophilic (Polar)
Carbon 3 (AI Predicted): Hydrophilic (Polar)
Oxygen 4  (Known Label): Hydrophilic (Polar)
